# 📞 Tool Calling and the Manual Execution Loop

## Learning Objectives
In this notebook, you will learn:
1. **Tool Calling Basics** - How a model decides when and which tools to invoke
2. **Binding Tools to an LLM** - Using `bind_tools()` to make tools available to a model
3. **Single vs. Parallel Tool Calls** - Reading the structured tool calls the model returns
4. **Manual Tool Execution Loop** - Building the full request → tool call → result → answer cycle with `ToolMessage`
5. **Prompt-Based Tool Calling** - Getting tool calls out of a model that has no native tool-calling support

## Prerequisites
- How tools are defined (`@tool`, descriptions, schemas) — covered in `01_defining_tools`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`. The tools here are deterministic stubs, so no other keys are needed.

---

## 📖 What is Tool Calling?

OpenAI's **tool calling** (previously called "function calling") lets a model decide to invoke one or more tools based on a user's prompt. Instead of generating a text answer, the model returns a structured JSON payload describing which tool to call and with what arguments.

### Key Concepts:
- **Tool selection is automatic** — the model reads tool descriptions and picks the right one
- **Multiple tools can fire in parallel** — a single prompt can trigger several tool calls at once
- **The model doesn't execute tools** — your code runs the tool and feeds results back to the model

> **Note**: OpenAI originally called this "function calling", then renamed it to "tool calling", and the terminology continues to evolve. Functionally they are the same. See the [official docs](https://platform.openai.com/docs/guides/function-calling).

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Configuration
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment configured successfully!")

✅ Environment configured successfully!


In [2]:
# ============================================================================
# LLM INITIALIZATION: Chat Model with Native Tool Calling
# ============================================================================
# gpt-4o-mini supports native tool calling, which Parts 3 and 4 depend on.
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---

## 🚫 Part 1: LLM Without Tools

Without tools, the model can only answer from its training data. It cannot access real-time information like weather, databases, or APIs. Let's see this limitation in action.

In [3]:
# ============================================================================
# TEST: LLM Response Without Tools
# ============================================================================

response = llm.invoke("How will the weather be in Munich today?")

print(f"🤖 Response (no tools):\n{response.content}")

🤖 Response (no tools):
I'm unable to provide real-time weather updates. I recommend checking a reliable weather website or app for the most current weather information in Munich.


---

## 🔧 Part 2: The Tools

Two deterministic stubs stand in for real APIs, so every run is reproducible. Their docstrings are what the model will read when it decides whether to call them (see `01_defining_tools` for how a tool becomes the schema the model sees).

In [4]:
# ============================================================================
# CUSTOM TOOLS: Define Weather and Seating Tools
# ============================================================================

from langchain_core.tools import tool


@tool
def fake_weather_api(city: str) -> str:
    """
    Check the weather in a specified city.

    Args:
        city (str): The name of the city where you want to check the weather.

    Returns:
        str: A description of the current weather in the specified city.
    """
    return "Sunny, 22°C"


@tool
def outdoor_seating_availability(city: str) -> str:
    """
    Check if outdoor seating is available at a specified restaurant in a given city.

    Args:
        city (str): The name of the city where you want to check for outdoor seating availability.

    Returns:
        str: A message stating whether outdoor seating is available or not.
    """
    return "Outdoor seating is available."


tools = [fake_weather_api, outdoor_seating_availability]

print(f"✅ Defined {len(tools)} tools: {[t.name for t in tools]}")

✅ Defined 2 tools: ['fake_weather_api', 'outdoor_seating_availability']


---

## 🔗 Part 3: Binding Tools to the LLM

Calling `llm.bind_tools(tools)` creates a new LLM instance that knows about our tools. When invoked, the model inspects the tool descriptions and decides whether to call one, multiple, or none based on the user's query.

In [5]:
# ============================================================================
# BIND TOOLS: Attach Tools to the LLM
# ============================================================================

llm_with_tools = llm.bind_tools(tools)

print("✅ Tools bound to LLM!")

✅ Tools bound to LLM!


### 🎯 Single Tool Call

When the query only relates to one tool, the model selects just that tool.

In [6]:
# ============================================================================
# TEST: Single Tool Call
# ============================================================================

result = llm_with_tools.invoke("How will the weather be in Munich today?")

print(f"🔧 Tool calls: {result.tool_calls}")
print(f"🤖 Content: '{result.content}'")

🔧 Tool calls: [{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_GRGobvpFuPRkrEUTERCcyNIw', 'type': 'tool_call'}]
🤖 Content: ''


### 🎯🎯 Multiple Tool Calls

When the query touches multiple tools, the model can fire them all in a single response — no extra round-trips needed.

In [7]:
# ============================================================================
# TEST: Multiple Tool Calls in One Request
# ============================================================================

result = llm_with_tools.invoke(
    "How will the weather be in Munich today? Do you still have seats outdoor available?"
)

print(f"🔧 Number of tool calls: {len(result.tool_calls)}")
for tc in result.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

🔧 Number of tool calls: 2
   → fake_weather_api({'city': 'Munich'})
   → outdoor_seating_availability({'city': 'Munich'})


### 🔍 Inspecting the Tool Call Response

The model returns an `AIMessage` with empty `content` and populated `tool_calls`. Let's inspect the full structure.

In [8]:
# ============================================================================
# INSPECT: Tool Call Structure
# ============================================================================

print("📋 tool_calls attribute:")
for tc in result.tool_calls:
    print(f"   name: {tc['name']}")
    print(f"   args: {tc['args']}")
    print(f"   id:   {tc['id']}")
    print()

print("📋 Full model_dump():")
result.model_dump()

📋 tool_calls attribute:
   name: fake_weather_api
   args: {'city': 'Munich'}
   id:   call_NncHRTcuzjhUUpVGfsYiAyfL

   name: outdoor_seating_availability
   args: {'city': 'Munich'}
   id:   call_SI1iHUwGSndcRfRKxrEr2cqU

📋 Full model_dump():


{'content': '',
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 51,
   'prompt_tokens': 165,
   'total_tokens': 216,
   'completion_tokens_details': {'accepted_prediction_tokens': 0,
    'audio_tokens': 0,
    'reasoning_tokens': 0,
    'rejected_prediction_tokens': 0,
    'text_tokens': None},
   'prompt_tokens_details': {'audio_tokens': 0,
    'cache_write_tokens': None,
    'cached_tokens': 0,
    'image_tokens': None,
    'text_tokens': None}},
  'model_provider': 'openai',
  'model_name': 'gpt-4o-mini-2024-07-18',
  'system_fingerprint': 'fp_55929dd0d0',
  'id': 'chatcmpl-ETnuFMNj3O0GEzx6ohK4ZqRO5EwmV',
  'service_tier': 'default',
  'finish_reason': 'tool_calls',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--01a0f255-1cf1-73a1-99d0-404f65addcaa-0',
 'tool_calls': [{'name': 'fake_weather_api',
   'args': {'city': 'Munich'},
   'id': 'call_NncHRTcuzjhUUpVGfsYiAyfL',
   'type': 'tool_call'},
  {'name': 'outdoo

---

## 🔄 Part 4: Manual Tool Execution Loop

The model doesn't execute tools itself — it only *requests* tool calls. Your code must:

1. Send the user message to the LLM
2. Read which tools the model wants to call
3. Execute each tool and collect results
4. Send everything back to the LLM for a final natural-language response

This is the core loop that agents automate under the hood.

In [9]:
# ============================================================================
# STEP 1: Send User Message and Get Tool Call Requests
# ============================================================================

from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        "How will the weather be in Munich today? I would like to eat outside if possible"
    )
]

llm_output = llm_with_tools.invoke(messages)
messages.append(llm_output)

print(f"🤖 LLM requested {len(llm_output.tool_calls)} tool call(s):")
for tc in llm_output.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

🤖 LLM requested 2 tool call(s):
   → fake_weather_api({'city': 'Munich'})
   → outdoor_seating_availability({'city': 'Munich'})


In [10]:
# ============================================================================
# STEP 2: Execute Tools and Append Results
# ============================================================================

tool_mapping = {
    "fake_weather_api": fake_weather_api,
    "outdoor_seating_availability": outdoor_seating_availability,
}

for tool_call in llm_output.tool_calls:
    tool_fn = tool_mapping[tool_call["name"].lower()]
    tool_output = tool_fn.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))
    print(f"🔧 {tool_call['name']} → {tool_output}")

🔧 fake_weather_api → Sunny, 22°C
🔧 outdoor_seating_availability → Outdoor seating is available.


In [11]:
# ============================================================================
# STEP 3: Get Final Response with Tool Results
# ============================================================================

final_response = llm_with_tools.invoke(messages)

print(f"🤖 Final Response:\n{final_response.content}")

🤖 Final Response:
The weather in Munich today is sunny with a temperature of 22°C. Additionally, outdoor seating is available, so you can enjoy your meal outside!


In [12]:
# ============================================================================
# INSPECT: Full Message History
# ============================================================================

print("📋 Complete message flow:")
for i, msg in enumerate(messages):
    print(f"   [{i}] {msg.__class__.__name__}: {msg.content[:80] or '(tool call request)'}")

📋 Complete message flow:
   [0] HumanMessage: How will the weather be in Munich today? I would like to eat outside if possible
   [1] AIMessage: (tool call request)
   [2] ToolMessage: Sunny, 22°C
   [3] ToolMessage: Outdoor seating is available.


---

## 📝 Part 5: Prompt-Based Tool Calling (For Models Without Native Support)

Everything above relied on **native** tool calling: the provider's API accepts tool schemas and returns structured `tool_calls`. Some older or open-weight models have no such support. For those, you can describe the tools in the prompt and ask the model to answer with a JSON blob, then parse it yourself.

It works, but you give up the guarantees of the native path: nothing forces the model to produce valid JSON, pick a real tool name or supply every argument. Prefer native tool calling whenever the model offers it.

In [13]:
# ============================================================================
# PROMPT-BASED TOOL CALLING: Describe the tools as plain text
# ============================================================================
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

# Render the same two tools as plain text for the prompt
rendered_tools = render_text_description(tools)
print("📋 Tool descriptions for the prompt:")
print("=" * 60)
print(rendered_tools)

📋 Tool descriptions for the prompt:
fake_weather_api(city: str) -> str - Check the weather in a specified city.

Args:
    city (str): The name of the city where you want to check the weather.

Returns:
    str: A description of the current weather in the specified city.
outdoor_seating_availability(city: str) -> str - Check if outdoor seating is available at a specified restaurant in a given city.

Args:
    city (str): The name of the city where you want to check for outdoor seating availability.

Returns:
    str: A message stating whether outdoor seating is available or not.


In [14]:
# ============================================================================
# PROMPT-BASED TOOL CALLING: Create the Prompt Template
# ============================================================================
# This prompt instructs the LLM to output its tool call as JSON

system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user instruction, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.
The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}")
])

print("✅ Prompt template created!")

✅ Prompt template created!


In [15]:
# ============================================================================
# PROMPT-BASED TOOL CALLING: Create the Chain and Execute
# ============================================================================
# Note: this chain uses the plain `llm`, not `llm_with_tools`: the tools reach
# the model only through the prompt text.

chain = prompt | llm | JsonOutputParser()

instructions = [
    {"input": "How will the weather be in Berlin today?"},
    {"input": "Is there outdoor seating in Hamburg?"},
]

# Run all instructions in parallel using map()
responses = chain.map().invoke(instructions)

print("🤖 LLM generated these tool calls:")
print("=" * 60)
for i, resp in enumerate(responses, 1):
    print(f"\n{i}. Tool: {resp['name']}")
    print(f"   Args: {resp['arguments']}")

🤖 LLM generated these tool calls:

1. Tool: fake_weather_api
   Args: {'city': 'Berlin'}

2. Tool: outdoor_seating_availability
   Args: {'city': 'Hamburg'}


In [16]:
# ============================================================================
# PROMPT-BASED TOOL CALLING: Execute the Tools
# ============================================================================
# The same tool_mapping from Part 4 dispatches the parsed calls

print("🔧 Executing tool calls:")
print("=" * 60)

for tool_call in responses:
    tool_fn = tool_mapping[tool_call["name"].lower()]
    tool_output = tool_fn.invoke(tool_call["arguments"])
    print(f"📞 {tool_call['name']}({tool_call['arguments']}) → {tool_output}")

🔧 Executing tool calls:
📞 fake_weather_api({'city': 'Berlin'}) → Sunny, 22°C
📞 outdoor_seating_availability({'city': 'Hamburg'}) → Outdoor seating is available.


---

## 📝 Summary

In this notebook, we learned:

### 1. Tool Calling Basics
- Without tools, LLMs cannot access real-time data
- **Tool calling** lets the model request external function execution via structured JSON; the model never runs the tool itself

### 2. Binding Tools
- `bind_tools()` attaches tool schemas to an LLM instance
- **Docstrings matter** — the model uses them to decide when to call each tool

### 3. Single vs. Parallel Tool Calls
- The model can request **one or many tools** in a single response
- Each tool call carries a unique `id` for matching its result back

### 4. Manual Tool Execution Loop
- The cycle: **User → LLM (tool requests) → your code runs the tools → LLM (final answer)**
- `ToolMessage` carries each result back to the model, matched by `tool_call_id`

### 5. Prompt-Based Tool Calling
- For models without native support: describe tools in the prompt and parse a JSON reply
- Less reliable than native tool calling; use it only when you must

### Next Steps
- **`03_augmented_llm_with_tools_langgraph`**: let LangGraph's `ToolNode` execute the calls for you